<a href="https://colab.research.google.com/github/acgibbs12/cosc-650-applied-llm-systems/blob/week-3/week3_prompt_engineering_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 (starter): Prompts as Engineering Artifacts

Runs without an API key: the semantic metric is local, and the model calls fall back to clearly-labeled fixtures so you can see the harness work. Set `GEMINI_API_KEY` to run the prompts for real. Cells marked **TODO (you)** are yours.

Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [1]:
import os, json, pathlib
from google.colab import userdata
import time

os.environ["COSCI650GeminiKey"] = userdata.get("COSCI650GeminiKey")


def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('COSCI650GeminiKey')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('COSCI650GeminiKey') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [2]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite
**TODO (you):** in your repo, store each prompt version as its own file. Here they are inline so the notebook runs. Task: classify a support ticket. v2 adds an intent rule.

In [3]:
PROMPT_V1 = 'Classify the ticket into one of: billing, technical, account, shipping. Return JSON {category, rationale}.'
PROMPT_V2 = (PROMPT_V1 + ' Classify by the primary intent, not incidental words: if money is only context for a delivery problem, choose shipping.')

tests = [
  {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
  {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
  {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
  {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
  {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
  {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
  {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
  {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
  {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
  {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
]
print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. The regression is your required failure.

In [4]:
# Labeled fixtures stand in for model output when LIVE is False. v2 fixes #7 but regresses #3.
FIX = {
  'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
  'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
}
def run_case(version, prompt, t):
    if LIVE:
        time.sleep(6)
        txt = gemini_chat([{'role':'user','content': prompt + '\nTicket: ' + t['ticket']}])
        try:
            start = txt.find('{')
            end = txt.rfind('}') + 1
            cleaned = txt[start:end]
            d = json.loads(cleaned)
            return d.get('category',''), d.get('rationale','')
        except Exception:
          return '', txt or ''
    return FIX[version][t['id']]

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat})
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {acc1:.0%}   v2 exact-match {acc2:.0%}')

v1 exact-match 90%   v2 exact-match 90%


In [5]:
for t in tests:
    e1 = next(r for r in r1 if r['id']==t['id'])['exact']
    e2 = next(r for r in r2 if r['id']==t['id'])['exact']
    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")
# TODO (you): explain why v2 helped one case and hurt another, and how you would resolve the tradeoff.

V2 improved case #7 because it helped the model focus on the main reason for the ticket. Even though the ticket mentions paying for express shipping, the actual problem is that the package arrived late and damaged. The new rule helped the model recognize that this was a shipping issue rather than a billing issue.

However, V2 caused an issue with case #3. The customer wants to change their email, which should be classified as an account issue, but they also mention that the save button does not work. V2 focused too much on the technical problem and classified it as technical instead of account.

To improve this, I would clarify that the model should focus on what the customer is trying to accomplish, while treating technical problems as secondary when they are only preventing the customer from completing that task.


In [6]:
txt = '''```json
{
  "category": "billing",
  "rationale": "The customer is complaining about being charged twice."
}
```'''

print("ORIGINAL:")
print(repr(txt))

start = txt.find('{')
end = txt.rfind('}') + 1
cleaned = txt[start:end]

print("\nCLEANED:")
print(repr(cleaned))

d = json.loads(cleaned)

print("\nCATEGORY:")
print(repr(d.get('category', '')))

print("\nRATIONALE:")
print(repr(d.get('rationale', '')))

ORIGINAL:
'```json\n{\n  "category": "billing",\n  "rationale": "The customer is complaining about being charged twice."\n}\n```'

CLEANED:
'{\n  "category": "billing",\n  "rationale": "The customer is complaining about being charged twice."\n}'

CATEGORY:
'billing'

RATIONALE:
'The customer is complaining about being charged twice.'


## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).